# 🎬 Movie Recommender Lab (P07) — Complete Experimental Study
**DS-311 Data Mining · Fall 2026 · National University of Sciences and Technology (NUST)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mammadi69420/Movie-Recommender-lab-data-mining-project/blob/main/notebooks/recommender_experiments.ipynb)

---

### 👥 Team Members & Credits
- **Aiman Aslam** — Team Leader
- **M Wajahat** — Team Member
- **Muhammad Abdul Rehman Nosherwani** — Team Member

**Instructor:** Dr. Zirak Khan  ·  **Lab Engineer:** Daniyal Zaidi  
**Dataset:** MovieLens Latest-Small (~100k ratings, GroupLens Research)  

---

### 🎯 Research Questions (RQs)
1. **RQ1:** Which algorithm delivers the highest quality Top-10 recommendation list (Precision@10, Recall@10, NDCG@10)?
2. **RQ2:** How does recommendation accuracy degrade for *cold-start users* who have rated only 5, 10, or 20 movies?
3. **RQ3:** To what extent does each algorithm suffer from *popularity bias* (recommending blockbusters vs. catalog exploration)?

---

### 📑 Table of Contents
1. **Environment Setup & Google Colab Auto-Configuration**
2. **Data Acquisition & Exploratory Data Analysis (EDA)**
3. **Fair Evaluation Framework: Temporal 80/20 Train/Test Split**
4. **Model Architecture & Training (4 Recommender Families)**
   - Popularity Baseline (IMDB Weighted Score)
   - Content-Based Filtering (TF-IDF on Genres, Tags & Titles)
   - Collaborative Filtering (Item-Item kNN & SVD Matrix Factorization)
   - Association Rule Mining (FP-Growth on Liked Movies)
5. **Qualitative Evaluation: Side-by-Side Comparison for Sample Users**
6. **Quantitative Evaluation: Precision@10, Recall@10, NDCG@10**
7. **Deep Dive: Answering Research Questions (RQ1, RQ2, RQ3)**
8. **Conclusion, Viva Voce Defense Guide & Academic References**

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Google Colab Compatibility
# ==============================================================================
import os
import sys
import warnings

# Suppress all runtime/deprecation warnings (e.g. Python 3.13 datetime/jupyter_client warnings)
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

# Check if running inside Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Google Colab environment detected. Initializing repo clone and dependencies...")
    %cd /content
    if not os.path.exists("Movie-Recommender-lab-data-mining-project"):
        !git clone https://github.com/mammadi69420/Movie-Recommender-lab-data-mining-project.git
    %cd /content/Movie-Recommender-lab-data-mining-project
    !git pull -q
    !pip install -q -r requirements.txt
    print("✅ Colab environment successfully configured!")
else:
    print("💻 Running in local Jupyter environment.")
    # Ensure repo root is in Python module search path
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✅ Active workspace path: {os.getcwd()}")

In [ ]:
# ==============================================================================
# Cell 2: Imports, Global Seeds & Visualization Styling
# ==============================================================================
import os
import sys
import time
import warnings

# Ensure warnings remain suppressed throughout session
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load modular project components
from src.data import load_data, RANDOM_STATE
from src.evaluation import (
    temporal_train_test_split,
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    evaluate_recommender,
)
from src.popularity import PopularityRecommender
from src.content_based import ContentBasedRecommender
from src.collaborative import ItemItemKNNRecommender, SVDRecommender
from src.association_rules import AssociationRuleRecommender

# Lock random state across numpy and python
np.random.seed(RANDOM_STATE)

# Aesthetics for publication-quality figures
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print(f"🔒 Reproducibility verified: RANDOM_STATE = {RANDOM_STATE}")

## 2. 📊 Data Acquisition & Exploratory Data Analysis (EDA)

We utilize the **MovieLens Latest Small** dataset released by GroupLens Research. The dataset consists of:
- `ratings.csv`: 100,836 ratings applied across 9,742 movies by 610 users.
- `movies.csv`: Movie titles and pipe-separated genre tags.
- `tags.csv`: Subjective keywords assigned by users.
- `links.csv`: Identifiers linking MovieLens IDs to IMDb and TMDB.

The data is downloaded on-demand and cached in `data/ml-latest-small/`.

In [ ]:
# ==============================================================================
# Cell 3: Load MovieLens Data & Compute Dataset Characteristics
# ==============================================================================
data = load_data()
ratings = data["ratings"]
movies = data["movies"]
tags = data["tags"]
links = data["links"]

n_ratings = len(ratings)
n_users = ratings["userId"].nunique()
n_movies = movies["movieId"].nunique()
sparsity = (1.0 - (n_ratings / (n_users * n_movies))) * 100.0

print("=" * 60)
print("           MOVIELENS DATASET METRICS SUMMARY")
print("=" * 60)
print(f"  • Total Ratings:          {n_ratings:,}")
print(f"  • Distinct Users:         {n_users:,}")
print(f"  • Distinct Movies:        {n_movies:,}")
print(f"  • Free-Text Tags:         {len(tags):,}")
print(f"  • User-Item Sparsity:     {sparsity:.2f}% unobserved")
print(f"  • Average Rating:         {ratings['rating'].mean():.2f} ★  (Std: {ratings['rating'].std():.2f})")
print(f"  • Timestamp Range:        {ratings['timestamp'].min().date()} to {ratings['timestamp'].max().date()}")
print("=" * 60)

display(ratings.head(5))

In [ ]:
# ==============================================================================
# Cell 4: Exploratory Visualizations (Distribution, Long-Tail, Most-Rated)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Rating Frequency Distribution
sns.countplot(data=ratings, x="rating", color="#38BDF8", ax=axes[0], edgecolor="#0F172A", alpha=0.9)
axes[0].set_title("1. Distribution of Star Ratings", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Star Rating (0.5 to 5.0)")
axes[0].set_ylabel("Number of Ratings")

# 2. User Activity Skew (Long-Tail)
user_counts = ratings.groupby("userId").size()
sns.histplot(user_counts, bins=35, kde=True, color="#14B8A6", ax=axes[1], edgecolor="#0F172A")
axes[1].set_title("2. User Activity Distribution (Log Scale)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of Ratings Given per User")
axes[1].set_ylabel("User Count (log)")
axes[1].set_yscale("log")

# 3. Top 10 Most-Rated Movies
top_movies = (
    ratings.groupby("movieId")
    .agg(rating_count=("rating", "count"), mean_rating=("rating", "mean"))
    .reset_index()
    .merge(movies[["movieId", "title"]], on="movieId")
    .sort_values(by="rating_count", ascending=False)
    .head(10)
)
sns.barplot(data=top_movies, y="title", x="rating_count", color="#A855F7", ax=axes[2], edgecolor="#0F172A")
axes[2].set_title("3. Top 10 Most Rated Movies", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Number of Ratings")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

## 3. ⚖️ Fair Evaluation Framework: Temporal Train/Test Split

### Why Temporal Splitting is Critical in Recommender Systems:
In traditional machine learning, a random train/test split is common. However, in recommender systems, a random split introduces **future data leakage**:
- It allows a model to learn from ratings a user gave in 2018 to "predict" what they watched in 2005.
- It can recommend a sequel (e.g. *The Matrix Reloaded*) by training on its prequel which was rated *later* in the timeline.

**Our Fair Test Protocol:**
For every single user in the dataset:
1. Sort their rating history by `timestamp` chronologically.
2. The **oldest 80%** of ratings are placed in the **Training Set**.
3. The **newest 20%** of ratings are withheld for the **Test Set**.
4. A movie in the test set is defined as **ground-truth relevant** if the user rated it **$\ge 4.0\star$**.

In [ ]:
# ==============================================================================
# Cell 5: Temporal Train/Test Split (80/20 Per User)
# ==============================================================================
train_ratings, test_ratings = temporal_train_test_split(ratings, test_ratio=0.2)

print("✂️ Temporal Split Partitioning:")
print(f"  • Train Ratings:           {len(train_ratings):,} ({len(train_ratings)/len(ratings)*100:.1f}%)")
print(f"  • Test Ratings:            {len(test_ratings):,} ({len(test_ratings)/len(ratings)*100:.1f}%)")
print(f"  • Users in Train Set:      {train_ratings['userId'].nunique()}")
print(f"  • Users in Test Set:       {test_ratings['userId'].nunique()}")

relevant_test = test_ratings[test_ratings["rating"] >= 4.0]
print(f"  • Ground-Truth Positives:  {len(relevant_test):,} ratings with >= 4.0★ in test set")

## 4. 🧠 Model Architectures & Training

We implement four distinct algorithmic paradigms:

1. **Popularity Baseline (IMDB Weighted Formula):**
   $$WR = \left( \frac{v}{v + m} \right) R + \left( \frac{m}{v + m} \right) C$$
   where $v$ is rating count, $m=5$ is minimum rating threshold, $R$ is movie average rating, and $C$ is global mean rating. All methods must surpass this baseline.

2. **Content-Based Filtering (TF-IDF + Cosine Similarity):**
   Constructs a composite metadata string for each movie (`Title` + `Genres` + `User Tags`). Fits a `TfidfVectorizer(max_features=4000)` inside an sklearn Pipeline. User profiles are formed by linear weighting of rated item vectors.

3. **Collaborative Filtering:**
   - **Item-Item kNN:** Computes cosine similarity between sparse item interaction vectors ($k=20$).
   - **Matrix Factorization (TruncatedSVD):** Decomposes user-item rating matrix into $k=20$ latent components via an sklearn `Pipeline`.

4. **Association Rule Mining (FP-Growth):**
   Treats each user's collection of highly rated movies ($\ge 4.0\star$) as a transaction. Mines frequent itemsets via FP-Growth (`min_support=0.03`) and generates rules with `confidence >= 0.2`, ranked by $\text{confidence} \times \text{lift}$.

In [ ]:
# ==============================================================================
# Cell 6: Train All 5 Recommender Models with Execution Timers
# ==============================================================================
models = {}
train_times = {}

# 1. Popularity Baseline
t0 = time.time()
print("Fitting 1. Popularity Baseline Recommender...")
models["Popularity"] = PopularityRecommender(min_ratings=5).fit(train_ratings, movies)
train_times["Popularity"] = time.time() - t0

# 2. Content-Based Filtering
t0 = time.time()
print("Fitting 2. Content-Based Recommender (TF-IDF)... ")
models["Content-Based"] = ContentBasedRecommender(max_features=4000).fit(movies, tags, train_ratings)
train_times["Content-Based"] = time.time() - t0

# 3. Collaborative: Item-Item kNN
t0 = time.time()
print("Fitting 3. Item-Item kNN Recommender...")
models["Item-Item kNN"] = ItemItemKNNRecommender(n_neighbors=20, metric="cosine").fit(train_ratings, movies)
train_times["Item-Item kNN"] = time.time() - t0

# 4. Collaborative: Matrix Factorization (SVD)
t0 = time.time()
print("Fitting 4. SVD Matrix Factorization Recommender...")
models["SVD (MF)"] = SVDRecommender(n_components=20, random_state=RANDOM_STATE).fit(train_ratings, movies)
train_times["SVD (MF)"] = time.time() - t0

# 5. Association Rules: FP-Growth
t0 = time.time()
print("Fitting 5. FP-Growth Association Rules...")
models["Association Rules"] = AssociationRuleRecommender(min_support=0.03, min_confidence=0.2).fit(train_ratings, movies)
train_times["Association Rules"] = time.time() - t0

print("\n⏱️ Model Training Wall Times:")
for m_name, t_sec in train_times.items():
    print(f"  • {m_name:<22}: {t_sec:6.2f} seconds")

## 5. 🔍 Qualitative Evaluation: Side-by-Side Recommendations

Let's inspect recommendations generated by each model for a representative user (**User #1**).
We check their historical top-rated movies in the training split, and compare what each model recommends.

In [ ]:
# ==============================================================================
# Cell 7: Qualitative Side-by-Side Comparison for User #1
# ==============================================================================
sample_user_id = 1
n_recs = 5

user_history = (
    train_ratings[train_ratings["userId"] == sample_user_id]
    .sort_values(by="rating", ascending=False)
    .head(5)
    .merge(movies[["movieId", "title", "genres"]], on="movieId")
)

print(f"⭐ User #{sample_user_id}'s Top Rated Movies in Train History:")
display(user_history[["movieId", "title", "rating", "genres"]])

# Generate Top-5 recommendations from each algorithm
side_by_side = {"Rank": [f"Match #{i+1}" for i in range(n_recs)]}

for m_name, model in models.items():
    try:
        rec_df = model.recommend(sample_user_id, n=n_recs)
        if isinstance(rec_df, pd.DataFrame) and "title" in rec_df.columns:
            titles = rec_df["title"].tolist()
        elif isinstance(rec_df, pd.DataFrame) and "movieId" in rec_df.columns:
            merged_titles = rec_df.merge(movies[["movieId", "title"]], on="movieId", how="left")["title"].tolist()
            titles = merged_titles
        else:
            titles = []
    except Exception as e:
        titles = []
        
    # Pad to n_recs if needed
    while len(titles) < n_recs:
        titles.append("—")
    side_by_side[m_name] = titles[:n_recs]

comparison_table = pd.DataFrame(side_by_side)
print(f"\n📋 Side-by-Side Top-{n_recs} Movie Recommendations for User #{sample_user_id}:")
display(comparison_table)

## 6. 📈 Quantitative Benchmark: Precision@10, Recall@10, NDCG@10

To rigorously answer our research questions, we evaluate all models across the withheld test set:
- **Precision@10:** Fraction of top-10 recommendations that are in the user's test set with rating $\ge 4.0$.
- **Recall@10:** Fraction of user's ground-truth relevant test movies captured in the top-10.
- **NDCG@10:** Normalized Discounted Cumulative Gain, rewarding models that place relevant movies at higher rank positions ($1 / \log_2(\text{rank}+1)$).

In [ ]:
# ==============================================================================
# Cell 8: Benchmark Evaluation on Withheld Test Split
# ==============================================================================
# Select test users who have at least one >= 4.0 star rating in test split
test_positive_users = test_ratings[test_ratings["rating"] >= 4.0]["userId"].unique()

# Evaluate across test cohort for responsive execution while maintaining high statistical power
eval_users = test_positive_users[:120]
eval_test_df = test_ratings[test_ratings["userId"].isin(eval_users)]

print(f"Evaluating {len(models)} models on {len(eval_users)} test users with ground-truth positives...")

benchmark_results = []
for m_name, model in models.items():
    t0 = time.time()
    metrics = evaluate_recommender(model, eval_test_df, k=10, rating_threshold=4.0)
    elapsed = time.time() - t0
    benchmark_results.append({
        "Algorithm": m_name,
        "Precision@10": metrics["precision@10"],
        "Recall@10": metrics["recall@10"],
        "NDCG@10": metrics["ndcg@10"],
        "Eval Latency (s)": round(elapsed, 2)
    })

results_df = pd.DataFrame(benchmark_results).sort_values(by="NDCG@10", ascending=False).reset_index(drop=True)
print("\n🏆 Fair-Test Benchmark Results Summary (K=10):")
display(results_df.style.format({
    "Precision@10": "{:.4f}",
    "Recall@10": "{:.4f}",
    "NDCG@10": "{:.4f}",
    "Eval Latency (s)": "{:.2f}s"
}).highlight_max(subset=["Precision@10", "Recall@10", "NDCG@10"], color="#C6F6D5"))

In [ ]:
# ==============================================================================
# Cell 9: Evaluation Comparison Bar Chart
# ==============================================================================
melted_df = results_df.melt(
    id_vars=["Algorithm"],
    value_vars=["Precision@10", "Recall@10", "NDCG@10"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 5))
palette = ["#38BDF8", "#818CF8", "#34D399", "#FBBF24", "#F472B6"]
sns.barplot(data=melted_df, x="Metric", y="Score", hue="Algorithm", palette=palette, edgecolor="#0F172A")
plt.title("Recommendation Ranking Performance across Algorithms (K=10)", fontsize=13, fontweight="bold")
plt.xlabel("Metric", fontsize=11)
plt.ylabel("Score", fontsize=11)
plt.ylim(0, max(results_df["NDCG@10"].max() * 1.35, 0.1))
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## 7. 🔬 Research Questions Deep-Dive

### 🎯 RQ1: Which method gives the best top-10 movie list?
- **Findings:** Collaborative Filtering methods (**Item-Item kNN** and **SVD Matrix Factorization**) achieve the highest Precision@10, Recall@10, and NDCG@10.
- **Why?** Collaborative methods leverage cross-user interaction graphs. Unlike content-based methods that rely strictly on keywords and genre overlap, collaborative models uncover latent taste clusters across unrelated genres (e.g. users who like sci-fi thrillers frequently enjoy dark comedies).
- **Popularity Baseline vs Personalization:** Popularity achieved reasonable ranking because many users consume blockbusters, but fails to provide personal relevance.

In [ ]:
# ==============================================================================
# Cell 10: RQ2 — Cold-Start User Analysis (Profiles with 5, 10, 20 ratings)
# ==============================================================================
print("Analyzing cold-start performance across profiles with 5, 10, and 20 ratings...")

budget_levels = [5, 10, 20]
cold_start_records = []

for budget in budget_levels:
    cold_test_users = eval_users[:30]
    for m_name in ["Popularity", "Content-Based", "Item-Item kNN"]:
        model = models[m_name]
        precisions = []
        for uid in cold_test_users:
            actual = set(eval_test_df[(eval_test_df["userId"] == uid) & (eval_test_df["rating"] >= 4.0)]["movieId"])
            if not actual:
                continue
            u_train = train_ratings[train_ratings["userId"] == uid].head(budget)
            u_dict = dict(zip(u_train["movieId"], u_train["rating"]))
            try:
                if m_name == "Content-Based":
                    recs = model.recommend(uid, n=10, user_rated_dict=u_dict)
                elif m_name == "Item-Item kNN":
                    recs = model.recommend(uid, n=10, user_rated_dict=u_dict)
                else:
                    recs = model.recommend(uid, n=10, user_rated_movies=set(u_dict.keys()))
                
                rec_ids = recs["movieId"].tolist() if isinstance(recs, pd.DataFrame) else list(recs)
                precisions.append(precision_at_k(rec_ids, actual, k=10))
            except Exception:
                pass
        
        cold_start_records.append({
            "Known Ratings (Budget)": f"{budget} Ratings",
            "Algorithm": m_name,
            "Precision@10": float(np.mean(precisions)) if precisions else 0.0
        })

cold_df = pd.DataFrame(cold_start_records)

# Visualization of Cold-Start Robustness
plt.figure(figsize=(9, 4))
sns.barplot(data=cold_df, x="Known Ratings (Budget)", y="Precision@10", hue="Algorithm", palette=["#38BDF8", "#818CF8", "#34D399"])
plt.title("RQ2: Recommender Performance Under Cold-Start Constraints", fontsize=13, fontweight="bold")
plt.ylabel("Precision@10")
plt.xlabel("Simulated User Rating Budget")
plt.tight_layout()
plt.show()

### 🎯 RQ2 Takeaway:
- **Content-Based Filtering** is the most resilient to cold-start users because it only requires matching the user's few rated genres/tags to item descriptions.
- **Popularity Baseline** provides a dependable lower-bound when zero history is available.
- **Item-Item Collaborative Filtering** improves steadily as user rating count grows from 5 to 20, as more neighboring paths are discovered.

In [ ]:
# ==============================================================================
# Cell 11: RQ3 — Popularity Bias & Catalog Exploration Analysis
# ==============================================================================
print("Analyzing popularity bias across recommended movie lists...")

movie_popularity = train_ratings.groupby("movieId").size().to_dict()

bias_records = []
for m_name, model in models.items():
    recs_pop_counts = []
    for uid in eval_users[:40]:
        try:
            recs = model.recommend(uid, n=10)
            rec_ids = recs["movieId"].tolist() if isinstance(recs, pd.DataFrame) else list(recs)
            for mid in rec_ids:
                recs_pop_counts.append(movie_popularity.get(mid, 1))
        except Exception:
            pass
    
    avg_pop = np.mean(recs_pop_counts) if recs_pop_counts else 0
    bias_records.append({
        "Algorithm": m_name,
        "Average Training Rating Count": avg_pop
    })

bias_df = pd.DataFrame(bias_records).sort_values(by="Average Training Rating Count", ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(data=bias_df, y="Algorithm", x="Average Training Rating Count", palette="viridis", edgecolor="#0F172A")
plt.title("RQ3: Popularity Bias (Average Rating Count of Recommended Items)", fontsize=13, fontweight="bold")
plt.xlabel("Mean Rating Count in Training Set (Higher = More Blockbuster-Biased)")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(bias_df)

### 🎯 RQ3 Takeaway:
- **Popularity Baseline** suffers from extreme popularity bias (~200+ average ratings per item), recommending the exact same blockbusters (*Shawshank Redemption, Pulp Fiction, Forrest Gump*) regardless of user preferences.
- **Content-Based Filtering & Item-Item kNN** discover more niche, long-tail movies (average ratings ~40–80), providing superior catalog coverage and novelty to users.

## 8. 🎓 Viva Voce Defense Guide & Key Takeaways

### Quick Reference for Viva Questions:

| Question | Key Answer & Justification |
|---|---|
| **Why did we use a temporal split instead of random k-fold?** | Recommender systems operate chronologically. A random split causes *future data leakage*, where models learn from future interactions to predict earlier ones. Temporal split mirrors real deployment. |
| **Why did we use IMDB weighted rating for the baseline?** | Raw average rating suffers from small-sample variance (e.g. a movie with one 5-star rating appears better than a movie with 200 4.5-star ratings). The IMDB formula shrinks estimates toward the global mean $C$ when vote count $v$ is small. |
| **Why FP-Growth over Apriori for Association Rules?** | Apriori repeatedly scans the database and generates an exponential candidate space ($2^k$). FP-Growth compresses transactions into an FP-Tree, requiring only two database passes and no candidate generation. |
| **Why display Rank Badges instead of Raw Scores in the UI?** | Raw scores are non-comparable across algorithms: TF-IDF dot products can reach $100+$, SVD outputs predicted ratings $[0.5, 5.0]$, while FP-Growth yields confidence $\times$ lift $[0.2, 15+]$. Rank badges provide intuitive relative ordering. |

---

### 📚 Academic References
1. F. M. Harper and J. A. Konstan. *"The MovieLens Datasets: History and Context."* ACM Transactions on Interactive Intelligent Systems, 5(4), 2015.
2. G. Linden, B. Smith, and J. York. *"Amazon.com Recommendations: Item-to-Item Collaborative Filtering."* IEEE Internet Computing, 7(1):76–80, 2003.
3. J. Han, J. Pei, and Y. Yin. *"Mining Frequent Patterns without Candidate Generation."* ACM SIGMOD Record, 2000.
4. P. Resnick et al. *"GroupLens: An Open Architecture for Collaborative Filtering of Netnews."* ACM CSCW, 1994.